In [36]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

datasets = {
    "full": "../../data/processed/ProcessedData.csv",
    "pca":  "../../data/reduced/dataPCA.csv",
    "fs":   "../../data/reduced/dataFS.csv"
}

outDir = "../../results/kmeans"

In [37]:
results = []
for name, path in datasets.items():
    
    df = pd.read_csv(path)
    
    if "Class" in df.columns:
        y = df["Class"].values
        X = df.drop(columns=["Class"])
    else:
        y = None
        X = df

    k = 4

    kmeans = KMeans(
        n_clusters=k,
        n_init=20,
        random_state=42
    )

    labels = kmeans.fit_predict(X)

    sil = silhouette_score(X, labels)
    db  = davies_bouldin_score(X, labels)

    results.append({
        "Dataset": name,
        "Algorithm": "KMeans",
        "K": k,
        "Silhouette": sil,
        "Davies_Bouldin": db
    })

    clusters = X.copy()
    clusters["cluster"] = labels
    clusters.to_csv(
        f"{outDir}/kmeans_{name}_clusters.csv",
        index=False
    )

    pca2D = PCA(n_components=2, random_state=42)
    X2d = pca2D.fit_transform(X)

    plt.figure(figsize=(6, 5))
    plt.scatter(
        X2d[:, 0], X2d[:, 1],
        c=labels, s=5, cmap="tab10"
    )
    plt.title(f"KMeans ({name})")
    plt.tight_layout()
    plt.savefig(f"{outDir}/kmeans_{name}.png", dpi=200)
    plt.close()

In [39]:
summaryData = pd.DataFrame(results)
summaryData.to_csv(f"{outDir}/kmeans_summary.csv", index=False)
print(summaryData)

  Dataset Algorithm  K  Silhouette  Davies_Bouldin
0    full    KMeans  4    0.377868        1.263575
1     pca    KMeans  4    0.380682        1.255122
2      fs    KMeans  4    0.365533        1.190992
